In [39]:
import pandas as pd
import numpy as np

df = pd.read_csv("Bengaluru_House_Data.xls")
df.head()
df.shape
print(df.isnull().sum())

area_type          0
availability       0
location           1
size              16
society         5502
total_sqft         0
bath              73
balcony          609
price              0
dtype: int64


In [40]:
df['total_sqft'] = df['total_sqft'].fillna("unknown")
df['bath'] = df['bath'].fillna(df['bath'].mean())
df['balcony'] = df['balcony'].fillna(df['balcony'].mean())
df['society'] = df['society'].fillna("Unknown")
df['location'] = df['location'].fillna("unknown")
df['size'] = df['size'].fillna("unknown")

print("Missing values after fill:")
print(df.isnull().sum())

Missing values after fill:
area_type       0
availability    0
location        0
size            0
society         0
total_sqft      0
bath            0
balcony         0
price           0
dtype: int64


In [41]:
df['total_sqft'] = df['total_sqft'].astype(str).str.extract('(\\d+)').astype(float)
df['total_sqft'] = df['total_sqft'].fillna(df['total_sqft'].mean())
print(df['total_sqft'].head())

0    1056.0
1    2600.0
2    1440.0
3    1521.0
4    1200.0
Name: total_sqft, dtype: float64


In [42]:
q1 = df['price'].quantile(0.25)
q3 = df['price'].quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

print(f"Before removing outliers: {len(df)} rows")
df = df[(df['price'] >= lower) & (df['price'] <= upper)]
print(f"After removing outliers: {len(df)} rows")

Before removing outliers: 13320 rows
After removing outliers: 12044 rows


In [43]:
df['bhk'] = df['size'].str.extract('(\\d+)').astype(float)
print(df[['size', 'bhk']].head())

        size  bhk
0      2 BHK  2.0
1  4 Bedroom  4.0
2      3 BHK  3.0
3      3 BHK  3.0
4      2 BHK  2.0


In [44]:
df['availability'] = np.where(df['availability'] == 'Ready To Move', 1, 0)
print(df['availability'].value_counts())

availability
1    9483
0    2561
Name: count, dtype: int64


In [45]:
df = df.drop(['size', 'society'], axis=1)
print(df.columns.tolist())
df = pd.get_dummies(df, columns=['area_type', 'location'])

['area_type', 'availability', 'location', 'total_sqft', 'bath', 'balcony', 'price', 'bhk']


In [48]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
df = df.fillna(0)
X = df.drop('price', axis=1)
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("="*50)
print("LINEAR REGRESSION - RESULTS")
print("="*50)
print(f"R² Score: {r2:.4f}")
print(f"RMSE: {rmse:.2f} Lakhs")

importance = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model.coef_
}).sort_values('Coefficient', key=abs, ascending=False)

print("\nTop 5 most important features:")
print(importance.head(5))

LINEAR REGRESSION - RESULTS
R² Score: 0.5149
RMSE: 29.17 Lakhs

Top 5 most important features:
                                               Feature  Coefficient
25                              location_1Channasandra  -174.443954
260                              location_Brigade Road   147.498255
24   location_12th cross srinivas nagar banshankari...   147.185866
875                             location_Postal Colony   144.743566
22                              location_1 Immadihalli  -141.960072
